# Neural Signal Denoising — Results Analysis

This notebook loads the trained model results and visualises:
- MSE comparison across FCN / RNN / LSTM
- Training loss curves
- Predicted vs actual signal plots
- Sensitivity analysis (OAT: learning rate sweep)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RESULTS_DIR = Path('../results')
DATA_DIR = Path('../data')

## 1. MSE Comparison Table

In [ ]:
df = pd.read_csv(RESULTS_DIR / 'comparison_table.csv')
df.style.highlight_min(subset=['train_mse', 'val_mse', 'test_mse'], color='lightgreen')

## 2. Bar Chart — Test MSE by Model

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(df))
ax.bar([r.upper() for r in df['model']], df['test_mse'], color=['red', 'orange', 'purple'])
ax.set_ylabel('Test MSE')
ax.set_title('Test MSE by Model')
plt.tight_layout()
plt.show()

## 3. Training Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, model in zip(axes, ['fcn', 'rnn', 'lstm']):
    log_path = RESULTS_DIR / f'training_log_{model}.csv'
    if log_path.exists():
        log = pd.read_csv(log_path)
        ax.plot(log['epoch'], log['train_mse'], label='Train')
        ax.plot(log['epoch'], log['val_mse'], label='Val')
        ax.set_title(model.upper())
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE')
        ax.legend()
plt.tight_layout()
plt.show()

## 4. Pre-generated Plots

In [ ]:
from IPython.display import Image, display

for fname in ['clean_noisy_predicted.png', 'mse_comparison.png', 'pred_vs_actual.png']:
    p = RESULTS_DIR / fname
    if p.exists():
        print(f'--- {fname} ---')
        display(Image(str(p)))

## 5. OAT Sensitivity — Learning Rate Sweep

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch.nn as nn
from neural_signal.services.sensitivity import SensitivityAnalyzer

base_cfg = {'learning_rate': 0.001, 'sensitivity_epochs': 10}
sa = SensitivityAnalyzer(base_cfg, RESULTS_DIR / 'sensitivity')

lr_values = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
results = sa.run(
    'learning_rate', lr_values,
    model_factory=lambda cfg: nn.Linear(15, 1),
    loader_factory=lambda cfg: SensitivityAnalyzer.make_simple_loaders(),
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx([r.param_value for r in results], [r.val_mse for r in results], 'o-')
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Val MSE')
ax.set_title('OAT Sensitivity: Learning Rate')
plt.tight_layout()
plt.show()